Importuojamos bibliotekos.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

import optuna
from lightgbm import LGBMRegressor
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean, RollingStd, RollingQuantile

Nustatomi keliai ir pagrindiniai parametrai.


In [ ]:
DATA_DIR = Path(r"...")
DATA_FILE = DATA_DIR / "main_lentele_50_su_features.parquet"
TEST_SET_FILE = DATA_DIR / "test_set_46.parquet"
OUTPUT_DIR = DATA_DIR / "lightgbm_results"

H = 7
N_WINDOWS = 26
VAL_DAYS = H * N_WINDOWS
TEST_DAYS = H * N_WINDOWS
OPTUNA_N_TRIALS = 60
RANDOM_STATE = 67
EXPECTED_FINAL_TEST_ROWS_46 = 46 * N_WINDOWS * H


Suskirstomi kintamieji į statinius, kategorinius ir kitus išorinius kintamuosius.

In [ ]:
ID_COMPONENTS = ["STORE_ID", "SKU_GROUP", "SKU_ID"]
STATIC_FEATURES = ["STORE_ID", "SKU_GROUP", "SKU_ID"]
DYNAMIC_FEATURES = [
    "PROMO_FLAG",
    "HOLIDAY_FLAG",
    "AVG_TEMP",
    "AVG_FEEL_TEMP",
    "TOTAL_PRECIP",
    "AVG_WIND",
    "YESTERDAY_HOLIDAY_FLAG",
    "TOMORROW_HOLIDAY_FLAG",
    "WEEKEND_FLAG",
    "DAY_OF_WEEK",
]
CATEGORICAL_FEATURES = ["STORE_ID", "SKU_GROUP", "SKU_ID", "DAY_OF_WEEK"]
METADATA_COLUMNS = ["sales_level", "zero_level", "series_type_2d", "mean_sales", "zero_pct"]


Įkeliami ir paruošiami duomenys.


In [ ]:
main = pd.read_parquet(DATA_FILE)
test_set_46 = pd.read_parquet(TEST_SET_FILE)

main["DATE"] = pd.to_datetime(main["DATE"])
test_set_46["DATE"] = pd.to_datetime(test_set_46["DATE"])

main["ds"] = main["DATE"]
main["y"] = main["SALES_QTY"].astype("float32")

main["unique_id"] = (
    main["STORE_ID"].astype(str)
    + "_"
    + main["SKU_GROUP"].astype(str)
    + "_"
    + main["SKU_ID"].astype(str)
)

Y_df_full = main[["unique_id", "ds", "y"] + STATIC_FEATURES + DYNAMIC_FEATURES].copy()
Y_df_full = Y_df_full.sort_values(["unique_id", "ds"]).reset_index(drop=True)

for col in CATEGORICAL_FEATURES:
    Y_df_full[col] = Y_df_full[col].astype("category")
    if col in test_set_46.columns:
        test_set_46[col] = test_set_46[col].astype("category")

selected_46_ids = sorted(test_set_46["unique_id"].unique())
selected_metadata_cols = ["unique_id"] + ID_COMPONENTS + [
    col for col in METADATA_COLUMNS if col in test_set_46.columns
]
selected_metadata = (
    test_set_46[selected_metadata_cols]
    .drop_duplicates(subset=["unique_id"])
    .reset_index(drop=True)
)

Sudaromos validavimo ir testinės aibių laiko ribos.

In [ ]:
max_date = Y_df_full["ds"].max()
min_date = Y_df_full["ds"].min()

test_start = max_date - pd.Timedelta(days=TEST_DAYS - 1)
val_start = test_start - pd.Timedelta(days=VAL_DAYS)
train_end = val_start - pd.Timedelta(days=1)
val_end = test_start - pd.Timedelta(days=1)

train_val_df = Y_df_full[Y_df_full["ds"] < test_start].copy()

Apibrėžiamos atsilikimų reikšmės ir slenkantys požymiai pardavimams.

In [ ]:
lags = [1, 2, 3, 7, 14, 21, 28]

lag_transforms = {
    1: [
        RollingMean(window_size=7),
        RollingMean(window_size=14),
        RollingMean(window_size=28),
        RollingQuantile(p=0.5, window_size=7),
        RollingQuantile(p=0.5, window_size=14),
        RollingQuantile(p=0.5, window_size=28),
        RollingStd(window_size=7),
        RollingStd(window_size=14),
        RollingStd(window_size=28),
    ]
}

Apibrėžiama WAPE paklaidos metrika.


In [ ]:
def wape(actual, predicted):
    actual = np.asarray(actual)
    predicted = np.asarray(predicted)
    actual_sum = np.sum(np.abs(actual))
    if actual_sum == 0:
        return np.nan
    return np.sum(np.abs(actual - predicted)) / actual_sum

Aprašomas LightGBM modelio ir MLForecast objekto kūrimas.


In [ ]:
LGBM_BASE_PARAMS = {
    "objective": "regression",
    "random_state": RANDOM_STATE,
    "verbosity": -1,
    "n_jobs": -1,
}

def suggest_lgbm_params(trial):
    return {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1000, step=100),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "num_leaves": trial.suggest_categorical("num_leaves", [15, 31, 63, 127]),
        "max_depth": trial.suggest_categorical("max_depth", [-1, 5, 8, 12]),
        "min_child_samples": trial.suggest_categorical("min_child_samples", [10, 20, 50, 100]),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 1.0),
    }

def make_lgbm(params):
    model_params = LGBM_BASE_PARAMS.copy()
    model_params.update(params)
    return LGBMRegressor(**model_params)

def make_forecaster(params):
    return MLForecast(
        models={"LightGBM": make_lgbm(params)},
        freq="D",
        lags=lags,
        lag_transforms=lag_transforms,
        num_threads=-1,
    )


Su Optuna ieškomi geriausi LightGBM parametrai.


In [ ]:
validation_trial_rows = []


def objective(trial):
    params = suggest_lgbm_params(trial)
    fcst = make_forecaster(params)
    val_cv_df = fcst.cross_validation(
        df=train_val_df,
        h=H,
        n_windows=N_WINDOWS,
        step_size=H,
        id_col="unique_id",
        time_col="ds",
        target_col="y",
        static_features=STATIC_FEATURES,
        refit=False,
    )
    wape_score = wape(val_cv_df["y"], val_cv_df["LightGBM"])
    row = {
        "trial": trial.number,
        **params,
        "validation_wape": wape_score,
        "validation_windows": val_cv_df["cutoff"].nunique(),
        "validation_rows": len(val_cv_df),
    }
    validation_trial_rows.append(row)

    if np.isnan(wape_score):
        return float("inf")
    return wape_score


study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)
study.optimize(objective, n_trials=OPTUNA_N_TRIALS, show_progress_bar=True)

validation_results = pd.DataFrame(validation_trial_rows).sort_values("trial").reset_index(drop=True)
validation_results.to_excel(OUTPUT_DIR / "lightgbm_validation_results.xlsx", index=False)

best_params = study.best_params.copy()
best_params_with_base = LGBM_BASE_PARAMS.copy()
best_params_with_base.update(best_params)

best_params_df = pd.DataFrame([
    {
        "best_trial": study.best_trial.number,
        "best_validation_wape": study.best_value,
        **best_params_with_base,
    }
])
best_params_df.to_excel(OUTPUT_DIR / "lightgbm_best_params.xlsx", index=False)

print("Best parameters:")
print(best_params_with_base)
print("Best validation WAPE:", study.best_value)

Suskaičiuojamos ir išsaugomos prognozės testavimo aibei naudojant geriausią atrinktą hiperparametrų kombinaciją.


In [ ]:
test_fcst = make_forecaster(best_params)
test_cv_df = test_fcst.cross_validation(
    df=Y_df_full,
    h=H,
    n_windows=N_WINDOWS,
    step_size=H,
    id_col="unique_id",
    time_col="ds",
    target_col="y",
    static_features=STATIC_FEATURES,
    refit=False,
)

test_cv_46 = test_cv_df[test_cv_df["unique_id"].isin(selected_46_ids)].copy()
test_cv_46 = test_cv_46.merge(selected_metadata, on="unique_id", how="left")
test_cv_46.to_parquet(OUTPUT_DIR / "lightgbm_test_predictions_46.parquet", index=False)